# 04 — FPS Benchmark(yolo26n/s-pose x GPU/CPU)

**用途**:對固定的一支 URFD 影片(300 幀,先全部解碼進記憶體)量測
`{yolo26n-pose, yolo26s-pose}` x `{GPU, GPU+FP16, CPU}` 的 FPS 與延遲。
只需要一個 **GPU runtime**——CPU 那一列是在同一個 session 裡把 `device`
強制指定成 `'cpu'` 跑出來的,不用另外開一個 CPU-only session。

**執行方式**:`Runtime → Run all`(GPU runtime,T4 即可)。**CPU 那幾組明顯較慢
是預期中的事**(Colab 只給 ~2 vCPU,數字本來就會偏低,誠實照報,不做美化)。
預估全部跑完 15-30 分鐘,大部分時間花在 CPU 推論上。

**方法論**(面試常被問):
- 影片先整支解碼進記憶體,benchmark 計時不受磁碟/解碼 I/O 影響。
- 每個設定先 warmup 20 幀(不計時)排除模型/CUDA 初始化開銷,GPU 計時前後
  夾 `torch.cuda.synchronize()`(CUDA 呼叫預設非同步,不同步會量到排隊時間)。
- 每個設定跑 3 輪取 FPS 中位數,抗單輪雜訊。
- 分報「純推論 FPS」(只算 `model.track()` 本身)與「端到端 FPS」(含
  tensor→numpy 轉換的實際開銷),另附 p50/p95 延遲。
- FP16(`quantize=16`)只在 GPU 有意義,CPU 不測這個維度。
- **不引用官方「CPU 快 43%」的宣傳數字**——那是 detect 模型的 ONNX 匯出
  數字,不適用本專案的 pose 模型與 PyTorch 直接推論。

跑完後把最後一格印出的 JSON 摘要貼回對話。


In [ ]:
!nvidia-smi


In [ ]:
import os
if os.path.basename(os.getcwd()) != 'fall-detection-pose':
    if not os.path.exists('fall-detection-pose'):
        !git clone -q https://github.com/tun0000/fall-detection-pose.git
    %cd fall-detection-pose
!pip -q install -e ".[infer]" pytest

# 同前幾本 notebook 的坑:pip install -e 之後 site.py 不會自動重新掃描 .pth,
# 這裡直接跑會 ModuleNotFoundError。reload(site) + 手動加 src/ 到 sys.path 雙重保險。
import importlib
import site
import sys

importlib.reload(site)
sys.path.insert(0, os.path.abspath("src"))

import fall_detection
print('fall_detection', fall_detection.__version__)


In [ ]:
# 規則引擎 + benchmark 純函式單元測試(純 CPU,~10 秒):必須全綠
!python -m pytest -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/fall-detection-pose'
DATA_DIR = f'{DRIVE_ROOT}/data/urfd'
OUT_DIR = f'{DRIVE_ROOT}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('沿用先前 notebook 的 Drive 路徑:', DRIVE_ROOT)


In [ ]:
# 固定用來計時的影片:adl-01(M2 已重組好的 mp4,沿用不必再下載)。
# 先整支解碼進記憶體,benchmark 計時才不受磁碟/解碼 I/O 影響。
from fall_detection.bench.benchmark import load_frames

VIDEO = f'{DATA_DIR}/videos/adl-01.mp4'
frames = load_frames(VIDEO, n_frames=300)
print(f'benchmark 影片:{VIDEO},載入 {len(frames)} 幀(要求 300)')


In [ ]:
# 跑完整矩陣:2 模型 x {GPU FP32, GPU FP16, CPU} = 6 組,每組 3 輪取中位數。
# CPU 那幾組明顯較慢是預期中的事,請耐心等待。
from fall_detection.bench.benchmark import benchmark

MODELS = ['yolo26n-pose.pt', 'yolo26s-pose.pt']
CONFIGS = [
    ('cuda:0', None),  # GPU FP32
    ('cuda:0', 16),    # GPU FP16
    ('cpu', None),     # CPU(強制指定,不用另開 CPU-only session)
]

results = []
for model_name in MODELS:
    for device, quantize in CONFIGS:
        print(f'跑 {model_name} / device={device} / quantize={quantize} ...')
        r = benchmark(frames, model_name=model_name, device=device, quantize=quantize, n_runs=3, warmup=20)
        results.append(r.to_dict())
        print(' ', r.to_dict())


In [ ]:
import json
import platform
import subprocess

import torch
import ultralytics


def _git_commit():
    try:
        return subprocess.run(
            ['git', 'rev-parse', '--short', 'HEAD'],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
    except Exception:
        return ''


report = {
    'video': VIDEO,
    'n_frames_requested': 300,
    'n_frames_used': len(frames),
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cpu': platform.processor() or platform.machine(),
    'cpu_count': os.cpu_count(),
    'git_commit': _git_commit(),
    'results': results,
}

with open(f'{OUT_DIR}/bench.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print('=== M5 摘要(請貼回對話) ===')
print(json.dumps(report, ensure_ascii=False, indent=2))
